In [4]:
import math
import time
import torch

DIM_MAX = 64
SEMENTES = (0, 1, 2)

# =============================================================================
# >>> SUA SUBMISSÃO — edite apenas esta classe <<<
# =============================================================================
class Submissao:
    DIM_MAX = 64

    def fit(self, X: torch.Tensor) -> None:
        g = torch.Generator().manual_seed(42)

        self.mu = X.mean(0)
        self.sd = X.std(0) + 1e-8
        Xs = (X - self.mu) / self.sd
        d = X.shape[1]

        # Estimar a distância mediana (escala base)
        amostras = min(X.shape[0], 300)
        d2 = torch.cdist(Xs[:amostras], Xs[:amostras]) ** 2
        med_val = d2[d2 > 0].median()
        med = med_val.item() if not torch.isnan(med_val) else 1.0
        if med < 1e-4:
            med = 1.0

        sigma = math.sqrt(med / 2)

        # Features explícitas que vamos usar
        self.use_cross = (d >= 2)
        n_cross = 1 if self.use_cross else 0

        # d originais + 1 (raio) + 1 (cross, opcional) = d + 2 features. O resto é RFF.
        n_rff = self.DIM_MAX - (d + 1 + n_cross)

        # 3 Escalas sem restrições: média, fina (para espiral) e grossa (para luas)
        escalas = [sigma, sigma * 0.25, sigma * 2.0]

        W_parts = []
        start = 0
        for i, sc in enumerate(escalas):
            size = (n_rff // len(escalas)) if i < len(escalas) - 1 else n_rff - start
            W_parts.append(torch.randn(d, size, generator=g) / sc)
            start += size

        self.W = torch.cat(W_parts, dim=1)
        self.b = torch.rand((n_rff,), generator=g) * 2 * math.pi
        self.rff_scale = math.sqrt(2.0 / n_rff)

    def phi(self, X: torch.Tensor) -> torch.Tensor:
        Xs = (X - self.mu) / self.sd

        # 1. Feature de Raio (Soma dos quadrados). 1 única dimensão mata círculos e esferas!
        raio_quadrado = (Xs ** 2).sum(dim=1, keepdim=True)

        parts = [Xs, raio_quadrado]

        # 2. Feature de Produto Cruzado. 1 única dimensão mata o XOR!
        if self.use_cross:
            cross = (Xs[:, 0] * Xs[:, 1]).unsqueeze(1)
            parts.append(cross)

        # 3. RFF para matar Luas e Espiral (usando float64 p/ blindar o determinismo)
        Xs_64 = Xs.double()
        W_64 = self.W.double()
        b_64 = self.b.double()

        rff_64 = self.rff_scale * torch.cos(Xs_64 @ W_64 + b_64)
        rff = rff_64.float() # Retorna pro float32 esperado

        parts.append(rff)
        return torch.cat(parts, dim=1)
# =============================================================================
# Harness (não edite daqui para baixo)
# =============================================================================
def _luas(n, g):
    t = torch.rand(n // 2, generator=g) * math.pi
    X = torch.cat([torch.stack([torch.cos(t), torch.sin(t)], 1),
                   torch.stack([1 - torch.cos(t), 0.5 - torch.sin(t)], 1)])
    y = torch.cat([torch.zeros(n // 2), torch.ones(n // 2)])
    return X + 0.15 * torch.randn(n, 2, generator=g), y

def _circulos(n, g):
    t = torch.rand(n, generator=g) * 2 * math.pi
    r = torch.where(torch.arange(n) < n // 2, 1.0, 0.45)
    X = torch.stack([r * torch.cos(t), r * torch.sin(t)], 1)
    return X + 0.08 * torch.randn(n, 2, generator=g), (torch.arange(n) >= n // 2).float()

def _xor(n, g):
    X = torch.rand(n, 2, generator=g) * 2 - 1
    y = (X[:, 0] * X[:, 1] < 0).float()
    return X + 0.15 * torch.randn(n, 2, generator=g), y

def _espiral(n, g):
    t = torch.sqrt(torch.rand(n // 2, generator=g)) * 3 * math.pi
    a = torch.stack([t * torch.cos(t), t * torch.sin(t)], 1) / 10
    y = torch.cat([torch.zeros(n // 2), torch.ones(n // 2)])
    return torch.cat([a, -a]) + 0.05 * torch.randn(n, 2, generator=g), y

def _esfera(n, g, d=10):
    X = torch.randn(n, d, generator=g)
    r2 = (X ** 2).sum(1)
    return X, (r2 > r2.median()).float()

TAREFAS = {"xor": lambda g: _xor(600, g), "duas_luas": lambda g: _luas(600, g),
           "circulos": lambda g: _circulos(600, g), "espiral": lambda g: _espiral(800, g),
           "esfera_10d": lambda g: _esfera(800, g)}

@torch.no_grad()
def perceptron_pocket(Z, y, epocas=50, eta=1.0, semente=0):
    Zb = torch.cat([Z, torch.ones(len(Z), 1)], 1)
    yb = 2 * y - 1
    w = torch.zeros(Zb.shape[1]); melhor_w, melhor_acc = w.clone(), -1.0
    g = torch.Generator().manual_seed(semente)
    for _ in range(epocas):
        for i in torch.randperm(len(Zb), generator=g).tolist():
            if yb[i] * (Zb[i] @ w) <= 0:
                w += eta * yb[i] * Zb[i]
        acc = ((Zb @ w) * yb > 0).float().mean().item()
        if acc > melhor_acc:
            melhor_acc, melhor_w = acc, w.clone()
    return melhor_w

def acuracia_balanceada(y, yhat):
    return torch.stack([(yhat[y == c] == c).float().mean() for c in y.unique()]).mean().item()

@torch.no_grad()
def rodar(sub, gerador, semente, checar=True):
    g = torch.Generator().manual_seed(semente)
    X, y = gerador(g)
    idx = torch.randperm(len(X), generator=g); ntr = int(0.6 * len(X))
    Xtr, ytr, Xte, yte = X[idx[:ntr]], y[idx[:ntr]], X[idx[ntr:]], y[idx[ntr:]]

    torch.manual_seed(semente)
    sub.fit(Xtr)
    t0 = time.perf_counter(); Ztr = sub.phi(Xtr); dt = time.perf_counter() - t0
    Zte = sub.phi(Xte)
    if checar:
        d, dl = Xtr.shape[1], Ztr.shape[1]
        assert Ztr.ndim == 2 and Zte.shape[1] == dl, "phi deve devolver (n, d')"
        assert d < dl <= DIM_MAX, f"exige d < d' <= {DIM_MAX}; recebi d={d}, d'={dl}"
        assert torch.isfinite(Ztr).all() and torch.isfinite(Zte).all(), "NaN/Inf na saída de phi"
        assert torch.allclose(sub.phi(Xtr[:20]), Ztr[:20]), "phi não é determinística"
        assert dt * (10_000 / len(Xtr)) < 2.0, "phi lenta demais"

    w = perceptron_pocket(Ztr, ytr, semente=semente)
    yhat = (torch.cat([Zte, torch.ones(len(Zte), 1)], 1) @ w > 0).float()
    return acuracia_balanceada(yte, yhat)

class _Identidade:
    def fit(self, X): pass
    def phi(self, X): return X

class _RFF:
    def fit(self, X):
        g = torch.Generator().manual_seed(0)
        self.mu, self.sd = X.mean(0), X.std(0) + 1e-8
        Xs = (X - self.mu) / self.sd
        d2 = torch.cdist(Xs[:300], Xs[:300]) ** 2
        sigma = math.sqrt(d2[d2 > 0].median().item() / 2)
        self.W = torch.randn(X.shape[1], DIM_MAX, generator=g) / sigma
        self.b = torch.rand(DIM_MAX, generator=g) * 2 * math.pi
    def phi(self, X):
        return math.sqrt(2 / DIM_MAX) * torch.cos(((X - self.mu) / self.sd) @ self.W + self.b)

def _mediana(cls, gerador, checar=True):
    vals = [rodar(cls(), gerador, sem, checar) for sem in SEMENTES]
    return float(torch.tensor(vals).median())

def avaliar():
    print(f"{'tarefa':<12}{'baseline':>10}{'referência':>12}{'você':>8}{'s_t':>7}")
    s = []
    for nome, gen in TAREFAS.items():
        b = _mediana(_Identidade, gen, checar=False)
        r = max(_mediana(_RFF, gen, checar=False), b + 1e-3)
        try:
            m = _mediana(Submissao, gen); erro = ""
        except AssertionError as e:
            m, erro = b, f"   <- {e}"
        st = min(max((m - b) / (r - b), 0.0), 1.25); s.append(st)
        print(f"{nome:<12}{b:>10.3f}{r:>12.3f}{m:>8.3f}{st:>7.2f}{erro}")
    S = 100 * (0.7 * sum(s) / len(s) + 0.3 * min(s))
    print(f"\nESCORE S = {S:.1f}   (0 = baseline, 100 = referência, até 125 com bônus)")
    return S

if __name__ == "__main__":
    avaliar()

tarefa        baseline  referência    você    s_t
xor              0.606       0.876   0.879   1.01
duas_luas        0.892       0.983   0.983   1.00
circulos         0.644       1.000   1.000   1.00
espiral          0.633       0.912   0.947   1.12
esfera_10d       0.508       0.872   0.969   1.25

ESCORE S = 105.4   (0 = baseline, 100 = referência, até 125 com bônus)
